# 210 · Hahn echo timing

**Level:** medium · **Topics:** dsl, sections, timing, Gen2 · **Type:** fix the bugs

A customer wrote a Hahn echo sequence:

`x90` – wait τ – `x180` – wait τ – `x90` – readout

The notebook looks complete, but it has **three bugs**:

- one makes the compilation fail,
- two are silent: the experiment runs, but the pulses are not where the customer expects.

Find and fix all three. Everything runs in emulation, no instrument needed.

In [ ]:
from laboneq.simple import *
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation

## Device setup

One SHFQC+ with one qubit: a drive line and a readout line.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFQC(uid="shfqc", address="DEV12001", device_options="SHFQC/PLUS/QC2CH")
)
setup.add_connections(
    "shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
)

## Calibration

📖 [Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html)

In [ ]:
calibration = Calibration()
calibration["q0/drive"] = SignalCalibration(
    oscillator=Oscillator(frequency=100e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=5.0e9),
    range=0,
)
calibration["q0/measure"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
calibration["q0/acquire"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
setup.set_calibration(calibration)

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## Pulses

📖 [Pulse library](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/tutorials/01_pulse_library.html)

In [ ]:
x90 = pulse_library.gaussian(length=100e-9, amplitude=0.5)
x180 = pulse_library.gaussian(length=100e-9, amplitude=1.0)
readout_pulse = pulse_library.const(length=1e-6, amplitude=0.5)

tau = 1e-6  # wait time before and after the x180 pulse

## Experiment

This is the customer's code. **The bugs are in this cell.**

📖 [Sections and pulses](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/index.html) ·
[Timing rules](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/01_timing_rules.html) ·
[Section tutorial](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/tutorials/00_section_tutorial.html)

In [ ]:
@dsl.experiment(signals=["drive", "measure", "acquire"])
def hahn_echo():
    with dsl.acquire_loop_rt(count=16):
        with dsl.section(name="first_x90"):
            dsl.play("drive", x90)

        # x180 in the middle of the sequence, a time tau after the first x90
        with dsl.section(name="echo_delay", length=tau):
            dsl.play("drive", x180)

        # wait another tau
        with dsl.section(name="second_delay", length=tau):
            pass

        with dsl.section(name="second_x90"):
            dsl.play("drive", x90)

        with dsl.section(name="readout", play_after="second_x90"):
            dsl.measure(
                measure_signal="measure",
                acquire_signal="acquire",
                measure_pulse=readout_pulse,
                integration_kernel=readout_pulse,
                handle="echo",
                reset_delay=1e-6,
            )

In [ ]:
experiment = hahn_echo()
experiment.set_signal_map(
    {
        "drive": "q0/drive",
        "measure": "q0/measure",
        "acquire": "q0/acquire",
    }
)

## Compile and run

📖 [Experiment](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/05_experiment/concepts/index.html)

In [ ]:
compiled = session.compile(experiment)
results = session.run(compiled)

## Check the timing

Plot the simulated output of the first shot.

📖 [Output simulator](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/10_advanced_topics/tutorials/00_output_simulator.html)

In [ ]:
plot_simulation(compiled, start_time=0, length=4e-6, signals=["drive", "measure"])

**Expected result** (τ = 1 µs, pulses of 100 ns):

| Pulse | Starts at |
| --- | --- |
| first `x90` | 0 µs |
| `x180` | 1.1 µs |
| second `x90` | 2.2 µs |
| readout | 2.3 µs |

## Questions

1. Why does the compilation fail? What does the error message tell you?
2. After fixing the error: when does the `x180` pulse start? Why?
3. When does the second `x90` start? Does the `second_delay` section do anything?
4. Fix the experiment so that the plot matches the expected result.

<details>
<summary>Hint 1</summary>

Print `experiment` and look at the `uid` of the sections. Is it the same as the `name`?
</details>

<details>
<summary>Hint 2</summary>

The `length` of a section sets the size of the window, not where the pulse sits inside it. Check the section `alignment`.
</details>

<details>
<summary>Hint 3</summary>

A section only occupies the signals that are used inside it. Which signals does an empty section use?
</details>

The solution is in [solution.ipynb](solution.ipynb).